# 06 — Heatmap temporal de pérdida por elemento

Visualización espacio-temporal: eje X = TestTime, eje Y = posición del elemento en la traza (km), color = pérdida estandarizada (Z-score por posición y ruta).

El Z-score por posición compara cada medición con el histórico de **ese mismo elemento físico**, por lo que destaca qué puntos de la traza están degradándose con independencia de su pérdida absoluta.

In [ ]:
import sys
import math
from pathlib import Path

SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors

from fms_client import FmsClient
import fms_extract as FE

client = FmsClient()

ROUTE_IDS = [41, 60, 104, 117]
TOP_PER_ROUTE = 300  # ajustar si se quiere más/menos historial

## 1. Extracción de pérdida por elemento

Para cada medición iOLM se expanden los `Elements[]` del `brief`: un registro por (medición × elemento), con su posición en km y su pérdida en dB. La pérdida que se usa es la del elemento puntual (`Results[].Loss`), **no** la de la sección de fibra previa — la fibra es una señal estructural estable; los conectores y splices son los que acumulan degradación.

In [ ]:
rows = []

for route_id in ROUTE_IDS:
    data = client.get_results(route_id, top=TOP_PER_ROUTE)
    for res in data.get("results", []):
        m = res.get("metadata", {})
        test_time = pd.to_datetime(m.get("TestTime"), errors="coerce", utc=True)
        route_name = m.get("AssetName", str(route_id))
        fault_status = m.get("FaultStatus", "")
        brief = res.get("brief", {}) or {}
        elements = (brief.get("Measurement") or {}).get("Elements") or []

        for el in elements:
            pos_m = FE._to_float(el.get("Position"))
            if math.isnan(pos_m):
                continue
            pos_km = pos_m / 1000
            el_type = el.get("Type", "")

            for r in el.get("Results") or []:
                loss = FE._to_float(r.get("Loss"))
                if math.isnan(loss):
                    continue
                rows.append({
                    "route_id":    route_id,
                    "route_name":  route_name,
                    "TestTime":    test_time,
                    "pos_km":      pos_km,
                    "loss_db":     loss,
                    "el_type":     el_type,
                    "fault_status": fault_status,
                })

df = pd.DataFrame(rows)
print(f"Total puntos (medición × elemento): {len(df):,}")
df.groupby("route_name").agg(
    mediciones=("TestTime", "nunique"),
    elementos_por_med=("pos_km", lambda x: round(len(x) / max(df.loc[x.index, "TestTime"].nunique(), 1), 1)),
    pos_min_km=("pos_km", "min"),
    pos_max_km=("pos_km", "max"),
)

## 2. Estandarización por posición

Cada elemento físico de la traza tiene su nivel base de pérdida (un conector en km 6.7 puede perder 0.24 dB de forma normal). El Z-score **por bin de posición dentro de cada ruta** elimina ese nivel base y deja visible solo la variación temporal: un Z alto indica que ese elemento está perdiendo más de lo habitual en ese punto.

In [ ]:
# Bin de 0.1 km para agrupar el mismo elemento físico entre mediciones
# (la posición puede variar ±algunos metros por resolución del OTDR)
df["pos_bin"] = (df["pos_km"] / 0.1).round() * 0.1

def zscore(s):
    std = s.std()
    return (s - s.mean()) / std if std > 0 else s * 0.0

df["loss_z"] = df.groupby(["route_id", "pos_bin"])["loss_db"].transform(zscore)

# Clip en ±3 σ para que los outliers extremos no colapsen la escala de color
CLIP = 3.0
df["loss_z_clip"] = df["loss_z"].clip(-CLIP, CLIP)

print("Z-score por ruta:")
df.groupby("route_name")["loss_z"].describe().round(3)

## 3. Gráfica espacio-temporal

Cada punto es un elemento medido en un instante dado. El color representa su Z-score de pérdida: violeta = por debajo del promedio histórico (sin degradación), rojo = muy por encima del promedio (degradación en ese punto de la traza).

In [ ]:
# Colormap rojo → naranja → amarillo → verde → azul → violeta
# (violeta = bajo Z = sin degradación; rojo = Z alto = degradación)
cmap = mcolors.LinearSegmentedColormap.from_list(
    "rv",
    ["violet", "blue", "cyan", "limegreen", "yellow", "orange", "red"],
)

routes_ordered = df.sort_values("route_id")["route_name"].unique()
n = len(routes_ordered)

fig, axes = plt.subplots(n, 1, figsize=(15, 4 * n), sharex=False)
if n == 1:
    axes = [axes]

for ax, route_name in zip(axes, routes_ordered):
    g = df[df["route_name"] == route_name].copy()

    sc = ax.scatter(
        g["TestTime"],
        g["pos_km"],
        c=g["loss_z_clip"],
        cmap=cmap,
        vmin=-CLIP,
        vmax=CLIP,
        s=18,
        alpha=0.75,
        linewidths=0,
    )

    cb = fig.colorbar(sc, ax=ax, pad=0.01, fraction=0.025)
    cb.set_label("Z-score pérdida (σ)", fontsize=9)
    cb.set_ticks([-3, -2, -1, 0, 1, 2, 3])

    ax.set_title(route_name, fontsize=11, fontweight="bold")
    ax.set_ylabel("Posición en traza (km)", fontsize=9)
    ax.set_xlabel("Tiempo de medición", fontsize=9)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=8)
    ax.yaxis.set_tick_params(labelsize=8)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

fig.suptitle(
    "Pérdida estandarizada por posición de elemento — Z-score histórico por posición y ruta\n"
    f"(top {TOP_PER_ROUTE} mediciones más recientes por ruta)",
    fontsize=12, y=1.01,
)
plt.tight_layout()

out = Path("../reports/figures/element_loss_heatmap.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Figura guardada en {out.resolve()}")
plt.show()

## 4. Variante: pérdida absoluta (dB) sin estandarizar

Útil para comparar el nivel real de pérdida entre posiciones — por ejemplo, identificar si un conector específico tiene pérdida crónicamente alta aunque no esté degradándose.

In [ ]:
fig2, axes2 = plt.subplots(n, 1, figsize=(15, 4 * n), sharex=False)
if n == 1:
    axes2 = [axes2]

for ax, route_name in zip(axes2, routes_ordered):
    g = df[df["route_name"] == route_name].copy()
    vmin, vmax = g["loss_db"].quantile(0.02), g["loss_db"].quantile(0.98)

    sc = ax.scatter(
        g["TestTime"],
        g["pos_km"],
        c=g["loss_db"],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=18,
        alpha=0.75,
        linewidths=0,
    )

    cb = fig2.colorbar(sc, ax=ax, pad=0.01, fraction=0.025)
    cb.set_label("Pérdida (dB)", fontsize=9)

    ax.set_title(route_name, fontsize=11, fontweight="bold")
    ax.set_ylabel("Posición en traza (km)", fontsize=9)
    ax.set_xlabel("Tiempo de medición", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=8)
    ax.yaxis.set_tick_params(labelsize=8)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

fig2.suptitle(
    "Pérdida absoluta por posición de elemento (dB)\n"
    f"(top {TOP_PER_ROUTE} mediciones más recientes por ruta)",
    fontsize=12, y=1.01,
)
plt.tight_layout()

out2 = Path("../reports/figures/element_loss_absolute.png")
plt.savefig(out2, dpi=150, bbox_inches="tight")
print(f"Figura guardada en {out2.resolve()}")
plt.show()